# Step 4: Model Registration
Upload the trained model to S3 and register it in RHOAI Model Registry.

In [ ]:
import boto3
import requests
import json
import os
from io import BytesIO

S3_ENDPOINT = os.environ.get("S3_ENDPOINT", "http://minio:9000")
S3_BUCKET = os.environ.get("S3_BUCKET", "pipelines")
AWS_ACCESS_KEY_ID = os.environ.get("AWS_ACCESS_KEY_ID", "minio")
AWS_SECRET_ACCESS_KEY = os.environ.get("AWS_SECRET_ACCESS_KEY", "minio123")

MODEL_KEY = os.environ.get("MODEL_KEY", "pipeline-artifacts/model.joblib")
METRICS_KEY = os.environ.get("METRICS_KEY", "pipeline-artifacts/metrics.json")
MODELS_BUCKET = os.environ.get("MODELS_BUCKET", "models")
MODEL_NAME = os.environ.get("MODEL_NAME", "loan-approval-classifier")
MODEL_VERSION = os.environ.get("MODEL_VERSION", "v1")
REGISTRY_URL = os.environ.get("REGISTRY_URL", "")

In [ ]:
s3 = boto3.client(
    "s3",
    endpoint_url=S3_ENDPOINT,
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    verify=False,
)

try:
    s3.head_bucket(Bucket=MODELS_BUCKET)
except Exception:
    s3.create_bucket(Bucket=MODELS_BUCKET)
    print(f"Created bucket: {MODELS_BUCKET}")

model_obj = s3.get_object(Bucket=S3_BUCKET, Key=MODEL_KEY)
model_data = model_obj["Body"].read()

dest_key = f"models/{MODEL_NAME}/{MODEL_VERSION}/model.joblib"
s3.put_object(Bucket=MODELS_BUCKET, Key=dest_key, Body=model_data)
s3_uri = f"s3://{MODELS_BUCKET}/{dest_key}"
print(f"Model copied to {s3_uri}")

In [ ]:
if REGISTRY_URL:
    metrics_obj = s3.get_object(Bucket=S3_BUCKET, Key=METRICS_KEY)
    metrics = json.loads(metrics_obj["Body"].read())

    resp = requests.post(
        f"{REGISTRY_URL}/api/model_registry/v1alpha3/registered_models",
        json={"name": MODEL_NAME, "description": "Loan approval classifier (RandomForest)"},
        verify=False,
    )
    model_id = resp.json().get("id")
    print(f"Registered model: {MODEL_NAME} (id={model_id})")

    resp = requests.post(
        f"{REGISTRY_URL}/api/model_registry/v1alpha3/model_versions",
        json={
            "name": MODEL_VERSION,
            "registeredModelId": model_id,
            "customProperties": {
                "s3_uri": {"string_value": s3_uri},
                "framework": {"string_value": "sklearn"},
                "accuracy": {"string_value": str(metrics.get("accuracy", ""))},
                "f1_score": {"string_value": str(metrics.get("f1_score", ""))},
            },
        },
        verify=False,
    )
    print(f"Registered version: {MODEL_VERSION}")
    print(f"Metrics: {metrics}")
else:
    print("REGISTRY_URL not set -- skipping Model Registry registration")
    print(f"Model is available at: {s3_uri}")